# Compare the HCD to CytoSig data and analyze time series within CytoSig

Requires download of CytoSig data at:
https://cytosig.ccr.cancer.gov/download/

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import re
import math
import pandas as pd
import os
from utils.plotting_utils import setup_plotting_environment, cm_to_inches
from utils.color_constants import CELL_TYPE_COLORS

plot_folder = "../../outputs/cytosig/plots"
os.makedirs(plot_folder, exist_ok=True)

data_folder = "../../data/cytosig"
setup_plotting_environment()

In [ ]:
# info on cytokines
cytokine_info = pd.read_excel("../../annotations/cytokine_info.xlsx", index_col=0, sheet_name="all_cytokines")
cytokine_info_parse = cytokine_info[cytokine_info["parse_cytokine"]]

cytokine_info_order = pd.read_excel("../../annotations/cytokine_info.xlsx", index_col=0, sheet_name="suggested_order")
gene_to_cytokine = cytokine_info.set_index("gene")['gene_to_cytokine'].dropna().to_dict()
cytokine_to_gene = {v: k for k, v in gene_to_cytokine.items()}
ALL_CYTOKINES_ORDER = cytokine_info['gene_to_cytokine'].dropna().tolist()

CYTOKINE_ORDER = list(cytokine_info_order["name"])
CYTOKINE_ORDER_WITH_PBS = ["PBS"] + CYTOKINE_ORDER
SINGLE_FAMILY_ORDER = cytokine_info_order["DEG_plot_family"].unique()

In [ ]:
cytosig_meta = pd.read_csv(os.path.join(data_folder, "cytosig_meta_info.txt"), index_col=0, sep="\t")
cytosig_meta['Duration'] = cytosig_meta['Duration'].fillna('unknown')
cytosig_meta['Dose'] = cytosig_meta['Dose'].fillna('unknown')

cytosig_meta['origin_publication'] = cytosig_meta.index.str.split("@").str[2].str.split(".").str[0]
cytosig_meta['fingerprint'] = cytosig_meta['Treatment'] + "_" + cytosig_meta['Condition'] + "_" + cytosig_meta['Dose'] + "_" + cytosig_meta['origin_publication']

In [ ]:
_UNIT_TO_HOURS = {
    # minutes
    "m": 1 / 60,
    "min": 1 / 60,
    "mins": 1 / 60,
    "minute": 1 / 60,
    "minutes": 1 / 60,

    # hours
    "h": 1,
    "hr": 1,
    "hrs": 1,
    "hour": 1,
    "hours": 1,
    "hs": 1,

    # days
    "d": 24,
    "day": 24,
    "days": 24,

    # weeks
    "w": 24 * 7,
    "wk": 24 * 7,
    "wks": 24 * 7,
    "week": 24 * 7,
    "weeks": 24 * 7,

    # months, approximate
    "mon": 24 * 30,
    "month": 24 * 30,
    "months": 24 * 30,

    # years, approximate
    "y": 24 * 365,
    "yr": 24 * 365,
    "yrs": 24 * 365,
    "year": 24 * 365,
    "years": 24 * 365,
}

def _format_number(x):
    """Avoid labels like 6.0h."""
    if np.isclose(x, round(x)):
        return str(int(round(x)))
    return f"{x:g}"

def time_to_hours(x, bare_unit="h"):
    """
    Convert messy time labels to numeric hours.

    Examples:
    '0.5h' -> 0.5
    '30m' -> 0.5
    '3days' -> 72
    '1w' -> 168
    '4d-6h' -> 90
    '18+8' -> 26, assuming bare numbers are hours
    """
    if pd.isna(x):
        return np.nan

    s = str(x).strip().lower()
    s = s.replace(" ", "").replace("_", "").replace("−", "-")

    if s in {"", "nan", "none", "unknown"}:
        return np.nan

    # Matches: 4d, -6h, +8, 0.5h, etc.
    tokens = re.findall(r"([+-]?)(\d+(?:\.\d+)?)([a-z]*)", s)

    if not tokens:
        return np.nan

    # Make sure the regex consumed the full string
    rebuilt = "".join(sign + num + unit for sign, num, unit in tokens)
    if rebuilt != s:
        return np.nan

    total_hours = 0.0

    for sign, num, unit in tokens:
        unit = unit or bare_unit

        if unit not in _UNIT_TO_HOURS:
            return np.nan

        value = float(num) * _UNIT_TO_HOURS[unit]

        if sign == "-":
            total_hours -= value
        else:
            total_hours += value

    return total_hours


def clean_time_label(x, bare_unit="h", max_days_as_hours=4):
    """
    Convert messy time labels to canonical labels.

    By default:
    - <1h becomes minutes if possible: 0.5h -> 30m
    - whole days up to 4d become hours: 3d -> 72h
    - whole days above 4d stay days: 1w -> 7d, 1mon -> 30d
    - non-whole-day composites stay hours: 5d+4h -> 124h
    """
    hours = time_to_hours(x, bare_unit=bare_unit)

    if pd.isna(hours):
        return 'unknown'

    # minutes, e.g. 0.5h -> 30m
    if abs(hours) < 1:
        minutes = hours * 60
        return f"{_format_number(minutes)}m"

    # exact whole days
    if np.isclose(hours % 24, 0):
        days = hours / 24

        if days > max_days_as_hours:
            return f"{_format_number(days)}d"
        else:
            return f"{_format_number(hours)}h"

    # otherwise use hours
    return f"{_format_number(hours)}h"

cytosig_meta['Duration_Clean'] = cytosig_meta['Duration'].apply(clean_time_label)
cytosig_meta['Duration_raw'] = cytosig_meta['Duration']
cytosig_meta['Duration'] = cytosig_meta['Duration_Clean']

In [ ]:
cytosig = pd.read_csv(os.path.join(data_folder, 'cytosig.merge.gz'), sep="\t", compression='gzip')
print(cytosig.columns.isin(cytosig_meta.index).sum())
print((~cytosig.columns.isin(cytosig_meta.index)).sum())

cytosig_orig = cytosig.copy()

In [ ]:
# we want a 24h time point and at least one other time point
duration_by_pub = cytosig_meta.groupby('fingerprint')['Duration'].value_counts().unstack(fill_value=0).drop(columns=['unknown'], errors='ignore')
duration_by_pub_filter = duration_by_pub[(duration_by_pub['24h'] > 0) & ((duration_by_pub > 0).sum(axis=1) > 1)]
cytosig_meta_orig = cytosig_meta.copy()
cytosig_meta = cytosig_meta[cytosig_meta['fingerprint'].isin(duration_by_pub_filter.index)]

In [ ]:
# Only take one dose per publication (highest available)
def dose_to_numeric(dose_str):
    m = re.match(r"\s*([0-9]*\.?[0-9]+)\s*(.*)", dose_str)
    if m is None:
        return '', ''.join(filter(str.isalpha, dose_str))
    return m.group(1), ''.join(filter(str.isalpha, m.group(2)))

cytosig_meta['fingerprint_wo_dose'] = cytosig_meta['Treatment'] + "_" + cytosig_meta['Condition'] + "_" + cytosig_meta['origin_publication']

print(cytosig_meta.groupby(['origin_publication', 'Treatment', 'Condition'])['Dose'].nunique().value_counts())

final_retained = []
unique_pubs_all = []
unique_pubs_filtered = []
for fp in cytosig_meta['fingerprint_wo_dose'].unique():
    subset = cytosig_meta[cytosig_meta['fingerprint_wo_dose'] == fp]
    unique_pubs_all.append(subset['origin_publication'].iloc[0])
    subset_groups = subset[['Dose', 'Duration']].value_counts().unstack(fill_value=0)
    
    # are there any cases where different doses have unequal number of time points?
    for _, row in subset_groups.iterrows():
        if any(row == 0):
            print(f"Unequal time points for {fp} with dose {row.name}")
            raise ValueError("Unequal time points across doses, cannot determine which dose to keep.")
            
    doses = subset['Dose'].unique()
    numbers, units = zip(*[dose_to_numeric(d) for d in doses])
    doses = pd.DataFrame({'Dose': doses, 'Number': numbers, 'Unit': units})
    if len(doses) == 1:
        final_fingerprint = subset['fingerprint'].iloc[0]
    else:
        different_units = doses['Unit'].unique()
        if len(different_units) == 1:
            # if all doses have the same unit, keep the one with the highest number
            max_number = doses['Number'].astype(float).max()
            dose_to_keep = doses[doses['Number'].astype(float) == max_number]['Dose'].iloc[0]
            subset_to_keep = subset[subset['Dose'] == dose_to_keep]
            final_fingerprint = subset_to_keep['fingerprint'].iloc[0]
        else:
            doses_with_numbers = doses[doses['Number'] != '']
            different_units = doses_with_numbers['Unit'].unique()
            if len(different_units) == 1:
                max_number = doses_with_numbers['Number'].astype(float).max()
                dose_to_keep = doses_with_numbers[doses_with_numbers['Number'].astype(float) == max_number]['Dose'].iloc[0]
                subset_to_keep = subset[subset['Dose'] == dose_to_keep]
                final_fingerprint = subset_to_keep['fingerprint'].iloc[0]
            else:
                raise ValueError(f"Cannot determine which dose to keep for {fp} due to different units.")
            
    final_retained.append(final_fingerprint)

print(f"Unique publications before filtering: {len(set(unique_pubs_all))}")
print(f"Retained {len(final_retained)} conditions after dose filtering.")

In [ ]:
cytosig_meta = cytosig_meta[cytosig_meta['fingerprint'].isin(final_retained)]
cytosig = cytosig.loc[:, cytosig.columns.isin(cytosig_meta.index)]

In [ ]:
cytosig_columns = cytosig.columns
all_tuples = []
for col in cytosig_columns:
    curr_tuple = (
        cytosig_meta.loc[col, 'origin_publication'],
        cytosig_meta.loc[col, 'Treatment'],
        cytosig_meta.loc[col, 'Condition'],
        cytosig_meta.loc[col, 'Dose'],
        cytosig_meta.loc[col, 'Duration']
    )
    all_tuples.append(curr_tuple)
cytosig.columns = pd.MultiIndex.from_tuples(all_tuples)

In [ ]:
time_series = cytosig_meta[['origin_publication', 'Treatment', 'Condition', 'Dose']].value_counts().reset_index(name='count')

In [ ]:
def time_float_to_string(time):
    if time >= 24:
        return f"{time / 24:.1f}d"
    elif time >= 1:
        return f"{time:.1f}h"
    elif time >= 1/60:
        return f"{time * 60:.1f}m"
    else:
        return f"{time * 3600:.1f}s"

def order_time_columns(columns):
    return sorted(columns, key=lambda x: time_to_hours(x))

filtered_time_series = []
for i in range(len(time_series)):
    current_series = time_series.iloc[i]

    pub = current_series['origin_publication']
    treat = current_series['Treatment']
    cell = current_series['Condition']
    dose = current_series['Dose']

    series_vals = cytosig.loc[:, (pub, treat, cell, dose, slice(None))].droplevel(axis=1, level=[0,1,2,3])
    sorted_times = order_time_columns(series_vals.columns)
    series_vals = series_vals[sorted_times].dropna(axis=0, how='any')
    series_vals = series_vals.T.groupby(series_vals.columns).mean().T # if there are duplicate time points, average them

    # at least 100 changed genes at a least one time point
    impact_by_time = (series_vals.abs() > 1).sum(axis=0)
    if impact_by_time.max() < 100: # nothing much to see here, skip
        continue
    
    filtered_time_series.append({
        'origin_publication': pub,
        'Treatment': treat,
        'Condition': cell,
        'Dose': dose,
        'DataFrame': series_vals,
        'time_points': len(series_vals.columns)
    })
filtered_time_series = pd.DataFrame(filtered_time_series)
len(filtered_time_series)

In [ ]:
picked_series = [  # (publication, treatment, condition)
    ("GSE125365", "TGFB3", "H358 Lung"),
    ("GSE12493", "TGFB1", "Dermal Fibroblast"),
    ("GSE17708", "TGFB1", "A549"),
    ("GSE48400", "IFNB", "Huh7"),
    ("E-MTAB-1027", "TNFA", "Bronchus Epithelial"),
]
key = list(zip(filtered_time_series['origin_publication'], filtered_time_series['Treatment'], filtered_time_series['Condition']))
picked_indices = [key.index(p) for p in picked_series]

for idx in picked_indices:
    row = filtered_time_series.iloc[idx]
    pub = row['origin_publication']
    treat = row['Treatment']
    cell = row['Condition']
    dose = row['Dose']
    series_vals = row['DataFrame']

    ref_time = '24h'
    times = order_time_columns(series_vals.columns)

    fig, axes = plt.subplots(1, len(series_vals.columns), figsize=(0.6*(1 + len(series_vals.columns)), 1),sharey=True)
    axes = axes.flatten()
    min_val = series_vals.min(axis=None)*1.05
    max_val = series_vals.max(axis=None)*1.05

    for i, time in enumerate(times):
        ax = axes[i]
        x = series_vals[ref_time]
        y = series_vals[time]
        ax.scatter(x, y, s=3, edgecolor='none', alpha=0.5, rasterized=True)

        if i == math.floor(len(times) / 2.0):
            ax.set_xlabel(f"log2FC ({ref_time})")
        if i == 0:
            ax.set_ylabel(f"log2FC at time")
            
        r = np.corrcoef(x, y)[0, 1]

        ax.set_xlim(min_val, max_val)
        ax.set_ylim(min_val, max_val)
        ax.set_title(fr"{time}", fontsize=7)
        ax.text(0.05, 0.8, f'r={r:.2f}', transform=ax.transAxes)

    fig.suptitle(f"{treat} stimulation of {cell} ({pub})", fontsize=7.5)
    plt.subplots_adjust(hspace=0.05, top=0.65)
    plt.savefig(os.path.join(plot_folder, f"detailed_plots_{pub}_{treat}_{cell}_{dose.replace('/', '_')}.svg"), dpi=400)
    plt.show()

In [ ]:
filtered_time_series['impact'] = None
filtered_time_series['correlations'] = None
filtered_time_series['correlations_all'] = None
for row_i, row in filtered_time_series.iterrows():
    pub = row['origin_publication']
    treat = row['Treatment']
    cell = row['Condition']
    dose = row['Dose']
    series_vals = row['DataFrame']
    
    ref_time = '24h'
    remaining_times = [col for col in series_vals.columns if col != ref_time]
    
    impact = (series_vals.abs() > 1).sum(axis=0)
    corrs = series_vals.corrwith(series_vals[ref_time], axis=0)
    corrs_all = series_vals.corr()
    filtered_time_series.at[row_i, 'impact'] = impact
    filtered_time_series.at[row_i, 'correlations'] = corrs
    filtered_time_series.at[row_i, 'correlations_all'] = corrs_all

In [ ]:
def get_plot(series):
    impact = series['impact']
    corrs = series['correlations']
    cell = series['Condition']
    dose = series['Dose']
    treatment = series['Treatment']
    
    sorted_times = order_time_columns(impact.index)
    impact = np.log10(impact[sorted_times] + 1)
    corrs = corrs[sorted_times]
    
    all_corrs = series['correlations_all']
    fig, ax = plt.subplots(1, 1, figsize=(0.17 * (3+len(all_corrs)), 0.17 * (3+len(all_corrs))))

    all_corrs = all_corrs.loc[sorted_times, sorted_times]
    sns.heatmap(all_corrs, ax=ax, cmap="Reds", vmin=0, vmax=1, annot=False, fmt=".2f",
                square=True, cbar_kws={'label': 'Pearson r', 'shrink': 0.6})
    ax.tick_params(axis='both', length=0)
    ax.tick_params(axis='x', rotation=90)
    ax.tick_params(axis='y', rotation=0)
    plt.tight_layout()

    return fig, ax

for idx in picked_indices:
    fig, ax = get_plot(filtered_time_series.iloc[idx])
    cyto = filtered_time_series.iloc[idx]['Treatment']
    cell = filtered_time_series.iloc[idx]['Condition']
    plt.savefig(os.path.join(plot_folder, f"summary_plot_{idx}_{cell}_{cyto}.svg"))
    plt.show()

In [ ]:
marker_cytokines = ['IFNA', 'IFNG', 'IL4', 'TGFB1', 'TNFA', 'IFNB', 'IL10', 'IL3']
study_colors = plt.get_cmap("tab20").colors

for cyto in marker_cytokines:
    marker_series = filtered_time_series[filtered_time_series['Treatment'] == cyto].copy()

    impacts_list = []

    for _, series in marker_series.iterrows():
        impacts_list.append(series['impact'])

    impacts_list = pd.DataFrame(impacts_list)
    impacts_list = impacts_list.div(impacts_list.max(axis=1), axis=0)

    ordered_columns = order_time_columns(impacts_list.columns)
    impacts_list = impacts_list[ordered_columns]

    fig, ax = plt.subplots(
        figsize=(0.5 + 0.4 * len(impacts_list.columns), 2)
    )

    x = np.arange(len(impacts_list.columns))

    for row_i, (_, row) in enumerate(impacts_list.iterrows()):
        mask = row.notna()
        ax.plot(
            x[mask],
            row.values[mask],
            alpha=1,
            color=study_colors[row_i % len(study_colors)],
            marker='o',
            markersize=5
        )

    ax.set_xticks(x)
    ax.set_xticklabels(impacts_list.columns)

    ax.set_ylabel("Relative Impact")
    ax.set_xlabel("Time")
    ax.set_title(f"{cyto}, n={len(impacts_list)} series", fontsize=8)
    ax.set_ylim(0, 1.05)

    plt.tight_layout()
    plt.savefig(os.path.join(plot_folder, f"impact_plot_{cyto}.svg"), dpi=300)
    plt.show()

In [ ]:
time_point_binning = {
    '0-3h': (0, 3),
    '3-9h': (3, 9),
    '9-30h': (9, 30),
    '30-90h': (30, 90),
    '90h+': (90, float('inf')) 
}
time_order = list(time_point_binning.keys())

def get_time_bins(cols):
    bins = []
    for col in cols:
        time_in_hours = time_to_hours(col)
        for bin_name, (lower, upper) in time_point_binning.items():
            if lower <= time_in_hours < upper:
                bins.append(bin_name)
                break
    return bins

marker_cytokines = ['IFNA', 'IFNG', 'IL4', 'TGFB1', 'TNFA', 'IFNB', 'IL10', 'IL3']
study_colors = plt.get_cmap("tab20").colors

fig, axes = plt.subplots(2, 4, figsize=cm_to_inches((8, 4.5)), sharey=True, sharex=True)
axes = axes.flatten()
for cyto_i, cyto in enumerate(marker_cytokines):
    ax = axes[cyto_i]
    marker_series = filtered_time_series[filtered_time_series['Treatment'] == cyto].copy()

    impacts_list = []

    for _, series in marker_series.iterrows():
        impacts_list.append(series['impact'])

    impacts_list = pd.DataFrame(impacts_list)
    impacts_list = impacts_list.div(impacts_list.max(axis=1), axis=0)

    ordered_columns = order_time_columns(impacts_list.columns)
    impacts_list = impacts_list[ordered_columns]
    binned_columns = get_time_bins(impacts_list.columns)
    impacts_list.columns = binned_columns

    impacts_list = impacts_list.stack().reset_index(name='Relative Impact').rename(columns={'level_1': 'Time Bin'})

    mean_vals = np.array([impacts_list.loc[impacts_list["Time Bin"] == t, "Relative Impact"].mean() for t in time_order])
    std_vals = np.array([impacts_list.loc[impacts_list["Time Bin"] == t, "Relative Impact"].std() for t in time_order])

    x = np.arange(len(time_order))
    x_lookup = {t: i for i, t in enumerate(time_order)}

    has_data = ~np.isnan(mean_vals)

    ax.bar(
        x[has_data],
        mean_vals[has_data],
        yerr=std_vals[has_data],
        width=0.75,
        color='tab:blue',
        alpha=1,
        capsize=2,
        error_kw={'color': 'black', 'linewidth': 1},
        edgecolor='black'
    )

    rng = np.random.default_rng(0)
    jitter = 0.12

    point_x = impacts_list["Time Bin"].map(x_lookup).astype(float)
    point_x = point_x + rng.uniform(-jitter, jitter, size=len(point_x))

    ax.scatter(
        point_x,
        impacts_list["Relative Impact"],
        color='black',
        s=3,
        alpha=1,
        edgecolor='none'
    )

    ax.set_xlim(-0.5, len(time_order) - 0.5)
    ax.set_xticks(x)
    ax.set_xticklabels(time_order, rotation=90, ha='center')
    ax.tick_params(axis='x', length=0)
    if cyto_i % 4 == 0:
        ax.set_ylabel("Rel. # of genes\nwith abs(log2FC>1)")
    ax.set_title(f"{cyto}, n={len(marker_series)}", fontsize=6.5, y=0.95)
    ax.set_ylim(0, 1.2)

plt.subplots_adjust(wspace=0.13, hspace=0.3)
plt.savefig(os.path.join(plot_folder, f"binned_impact_plot.svg"), dpi=300)
plt.show()

In [ ]:
corr_records = []

for cyto in marker_cytokines:
    marker_series = filtered_time_series[filtered_time_series['Treatment'] == cyto].copy()

    fingerprints = (
        marker_series['origin_publication'].astype(str) + "_" +
        marker_series['Treatment'].astype(str) + "_" +
        marker_series['Condition'].astype(str) + "_" +
        marker_series['Dose'].astype(str)
    )

    for fingerprint, (_, series) in zip(fingerprints, marker_series.iterrows()):
        s = series['correlations_all'].copy()

        binned_times = pd.Series(get_time_bins(s.columns), index=s.columns)

        bin_names = list(time_point_binning)

        for i, row_bin in enumerate(bin_names):
            row_times = binned_times.index[binned_times == row_bin]

            for j, col_bin in enumerate(bin_names[i:], start=i):
                col_times = binned_times.index[binned_times == col_bin]

                sub_s = s.loc[row_times, col_times].copy()

                if sub_s.empty:
                    continue

                if row_bin == col_bin:
                    mask = np.triu(np.ones(sub_s.shape, dtype=bool), k=1)
                    vals = sub_s.where(mask).stack(future_stack=True).dropna()

                else:
                    vals = sub_s.stack(future_stack=True).dropna()

                comparison_type = f"{row_bin} to {col_bin}"

                for (time_1, time_2), corr_val in vals.items():
                    corr_records.append({
                        "marker_cytokine": cyto,
                        "fingerprint": fingerprint,
                        "comparison_type": comparison_type,
                        "time_1": time_1,
                        "time_2": time_2,
                        "corr": corr_val,
                    })

corr_by_fingerprint_df = pd.DataFrame(corr_records)

In [ ]:
for cyto in marker_cytokines:
    df = corr_by_fingerprint_df[corr_by_fingerprint_df['marker_cytokine'] == cyto].copy()

    bin_order = list(time_point_binning)
    bin_to_x = {b: i for i, b in enumerate(bin_order)}

    # recover the bins for each exact time
    df["time_1_bin"] = get_time_bins(df["time_1"])
    df["time_2_bin"] = get_time_bins(df["time_2"])

    df["x"] = df["time_1_bin"].map(bin_to_x)
    df["y"] = df["time_2_bin"].map(bin_to_x)

    rng = np.random.default_rng(0)
    jitter = 0.28

    df["x_jit"] = df["x"] + rng.uniform(-jitter, jitter, size=len(df))
    df["y_jit"] = df["y"] + rng.uniform(-jitter, jitter, size=len(df))

    fig, ax = plt.subplots(figsize=(4, 4))

    sc = ax.scatter(
        df["x_jit"],
        df["y_jit"],
        c=df["corr"],
        vmin=0,
        vmax=1,
        cmap="Reds",
        s=12,
        alpha=1,
        edgecolor="none"
    )

    ax.set_xticks(range(len(bin_order)))
    ax.set_yticks(range(len(bin_order)))
    ax.set_xticklabels(bin_order, rotation=45, ha="right")
    ax.set_yticklabels(bin_order)

    ax.set_xlim(-0.5, len(bin_order) - 0.5)
    ax.set_ylim(-0.5, len(bin_order) - 0.5)

    ax.set_title(f"{cyto} correlations by time-bin pair")
    ax.set_aspect("equal")

    # grid around bin-pair cells
    for b in np.arange(-0.5, len(bin_order), 1):
        ax.axvline(b, color="black", linewidth=0.5, alpha=0.25)
        ax.axhline(b, color="black", linewidth=0.5, alpha=0.25)

    cbar = fig.colorbar(sc, ax=ax, shrink=0.7)
    cbar.set_label("Correlation")

    plt.tight_layout()
    plt.savefig(os.path.join(plot_folder, f"correlation_by_time_bin_pair_{cyto}.svg"), dpi=300)

    # ----------------------------------------------------------------------------------------
    df = corr_by_fingerprint_df[
        corr_by_fingerprint_df["marker_cytokine"] == cyto
    ].copy()

    bin_order = list(time_point_binning)

    # Add explicit bins if not already present
    df["time_1_bin"] = pd.Series(get_time_bins(df["time_1"]), index=df.index)
    df["time_2_bin"] = pd.Series(get_time_bins(df["time_2"]), index=df.index)

    # Make sure comparison is always ordered: early-late, not late-early
    bin_rank = {b: i for i, b in enumerate(bin_order)}
    df["bin_a"] = df[["time_1_bin", "time_2_bin"]].apply(lambda x: x.iloc[0] if bin_rank[x.iloc[0]] <= bin_rank[x.iloc[1]] else x.iloc[1], axis=1)
    df["bin_b"] = df[["time_1_bin", "time_2_bin"]].apply(lambda x: x.iloc[1] if bin_rank[x.iloc[0]] <= bin_rank[x.iloc[1]] else x.iloc[0], axis=1)
    summary = df.groupby(["bin_a", "bin_b"], observed=True)["corr"].agg(mean="mean", sd="std", n="count").reset_index()

    mean_mat = pd.DataFrame(np.nan, index=bin_order, columns=bin_order)
    sd_mat = pd.DataFrame(np.nan, index=bin_order, columns=bin_order)
    n_mat = pd.DataFrame(0, index=bin_order, columns=bin_order)

    for _, row in summary.iterrows():
        mean_mat.loc[row["bin_a"], row["bin_b"]] = row["mean"]
        sd_mat.loc[row["bin_a"], row["bin_b"]] = row["sd"]
        n_mat.loc[row["bin_a"], row["bin_b"]] = row["n"]

    fig, ax = plt.subplots(figsize=cm_to_inches((6, 5)))

    im = ax.imshow(mean_mat.astype(float), vmin=-1, vmax=1, cmap="coolwarm")
    ax.set_xticks(np.arange(len(bin_order)))
    ax.set_yticks(np.arange(len(bin_order)))
    ax.set_xticklabels(bin_order, rotation=45, ha="right")
    ax.set_yticklabels(bin_order)

    ax.set_xlabel("Time bin 2")
    ax.set_ylabel("Time bin 1")
    ax.set_title(f"{cyto}", fontsize=7)

    # Hide lower triangle
    for i in range(len(bin_order)):
        for j in range(len(bin_order)):
            if j < i:
                ax.add_patch(
                    plt.Rectangle(
                        (j - 0.5, i - 0.5),
                        1,
                        1,
                        color="white",
                        zorder=3
                    )
                )

    # Annotate upper triangle
    for i, row_bin in enumerate(bin_order):
        for j, col_bin in enumerate(bin_order):
            if j < i:
                continue

            mean_val = mean_mat.loc[row_bin, col_bin]
            sd_val = sd_mat.loc[row_bin, col_bin]
            n_val = n_mat.loc[row_bin, col_bin]

            if pd.isna(mean_val):
                continue

            ax.text(
                j,
                i,
                f"{mean_val:.2f}\n±{sd_val:.2f}\nn={int(n_val)}" if pd.notna(sd_val) else \
                    f"{mean_val:.2f}\nn={int(n_val)}",
                ha="center",
                va="center",
                fontsize=5.5
            )
    ax.tick_params(axis='both', length=0)

    cbar = fig.colorbar(im, ax=ax, shrink=0.75, pad=0.03)
    cbar.set_label("Mean correlation")

    plt.tight_layout()
    plt.savefig(os.path.join(plot_folder, f"correlation_summary_by_time_bin_pair_{cyto}.svg"), dpi=300)
    plt.show()

## Compare to hcd

In [ ]:
from utils.processing_utils import get_protein_coding_genes

protein_coding_genes = get_protein_coding_genes(annotation_folder="../../annotations")
hcd = pd.read_parquet("../../outputs/main_screen/output_tables/raw_log2fc_values.parquet")
hcd = hcd[hcd.index.isin(protein_coding_genes)]

hcd_pbmc = pd.read_parquet("../../outputs/main_screen/output_tables/pbmc_weighted_log2fc.parquet")
hcd_pbmc = hcd_pbmc[hcd_pbmc.index.isin(protein_coding_genes)]

hcd_pbmc.columns = pd.MultiIndex.from_tuples(("PBMC", col) for col in hcd_pbmc.columns)
hcd = pd.concat([hcd, hcd_pbmc], axis=1)

In [ ]:
cell_type_mapping = {
    "Pbmc": "PBMC",
    "Purified Peripheral Blood": "PBMC",
    "Normal Control Pbmcs": "PBMC",
    "Pbmc.Healthy": "PBMC",
    "Pbmc From Patients With Multiple Sclerosis": "PBMC",
    "Peripheral Blood Mononuclear": "PBMC",
    "Pbmc.Complete_Responder": "PBMC",
    "Pbmc.Partial_Responder": "PBMC",
    "Whole Blood": "PBMC",
    "Whole Blood.Healthy Adults": "PBMC",
    "Whole Blood.Healthy Pediatric": "PBMC",

    "B": "B cell",
    "Naïve B Cells Purified From Peripheral Blood": "B cell",
    "Purified Naive B Cells From Blood": "B cell",
    "Primary B": "B cell",
    "Cd19+ B": "B cell",
    "Encode Ebv-Immortalized B": "B cell",
    "Normal B": "B cell",

    "T Cd4helper": "CD4 T cell",
    "Cd4-Positive, Alpha-Beta Memory T": "CD4 T cell",
    "T Cd4": "CD4 T cell",
    "T Cd4Cd25-": "CD4 T cell",
    "Activated Naive T Cd4": "CD4 T cell",
    "Cd4 Effector Memory T": "CD4 T cell",
    "Primary T Cd4": "CD4 T cell",
    "T": "CD4 T cell",
    "T Lymphocyte": "CD4 T cell",
    "Th0": "CD4 T cell",
    "Th1": "CD4 T cell",
    "Th2": "CD4 T cell",
    "Th17": "CD4 T cell",
    "Naïve T Cd4": "CD4 T cell",
    "T Cd4h": "CD4 T cell",
    "T Cd4 Cell_Activated Via Cd3Cd28": "CD4 T cell",
    "Primary Sorted Memory T Cd4 Cell Subsets Derived From Pbmcs.Cm": "CD4 T cell",
    "Primary Sorted Memory T Cd4 Cell Subsets Derived From Pbmcs.Em": "CD4 T cell",
    "Primary Sorted Memory T Cd4 Cell Subsets Derived From Pbmcs.Tm": "CD4 T cell",

    "Facs Purified Tregs": "Treg", 
    "Blood Tregs": "Treg",

    "T Cd8": "CD8 T cell",
    "T Cd8_With Anti Cd3/Cd8": "CD8 T cell",
    "Antigen Specific T Cd8 Cell Clone From Melanoma Patients": "CD8 T cell",
    "Ccr7-Cd45Ra+ T Cd8": "CD8 T cell",
    "Cd8Cd45Ra+": "CD8 T cell",

    "Macrophage.Uninfected": "CD14 Mono",
    "Macrophage": "CD14 Mono",
    "Monocyte": "CD14 Mono",
    "Monocyte Derived Macrophage": "CD14 Mono",
    "M1 Mdm_Cf": "CD14 Mono",
    "M1 Mdm_Noncf": "CD14 Mono",
    "Monocyte-Derived Macrophage": "CD14 Mono",
    "Cd14+ Monocyte": "CD14 Mono",
    "Ipsc-Derived Monocyte": "CD14 Mono",
    "Primary Monocyte-Derived Macrophage": "CD14 Mono",
    "Monocyte-Dervied Macrophage": "CD14 Mono",
    "Peripheral Blood- Isolated Cd14 Monocyte": "CD14 Mono",
    "Circulating Macrophage": "CD14 Mono",
    "Mac": "CD14 Mono",
    "Mono": "CD14 Mono",
    "Primary Macrophage": "CD14 Mono",
    "Cd14(+) Monocyte-Derived Macrophage": "CD14 Mono",
    "Cultured Macrophage": "CD14 Mono",
    "Primary Monocyte": "CD14 Mono",
    "Gmcsf Macrophage": "CD14 Mono",
    "Mcsf Macrophage": "CD14 Mono",
    "Activated Macrophage": "CD14 Mono",
    "Corrected Ipsc-Derived Macrophage": "CD14 Mono",
    "Disease Ipsc-Derived Macrophage": "CD14 Mono",
    "Wt Esc-Derived Macrophage": "CD14 Mono",
    "Blood Cd14+ Monocyte": "CD14 Mono",
    "Cd14+ Monocyte With Mcsf": "CD14 Mono",
    "Peripheral Blood Monocyte": "CD14 Mono",
    "Monocyte+Mcsf": "CD14 Mono",
    "Pbmc.Cd14+": "CD14 Mono",
    "Monocytes Stimulated With M-Csf": "CD14 Mono",
    "Monocytes Stimulated With Gm-Csf": "CD14 Mono",
    "Mdm": "CD14 Mono",
    "Blood-Derived Macrophage": "CD14 Mono",
    "Classical Monocytes.Adult": "CD14 Mono",
    "Classical Monocytes.Fetal": "CD14 Mono",
    "Monocyte Isolation From Buffy Coats": "CD14 Mono",
    "Cd14Pos_Monocyte": "CD14 Mono",
    "Cd14+ Peripheral Blood Monocyte": "CD14 Mono",
    
    "Primary Cd16+ Monocyte": "CD16 Mono",

    "Dendritic": "cDC",
    "Monocyte Derived Dendritic Cellc": "cDC",
    "Immature Dendritic": "cDC",
    "Monocyte Derived Immature Dcs": "cDC",
    "Monocyte-Derived Dendritic": "cDC",
    "Dc": "cDC",
    "Immature Dc": "cDC",
    "Primary Blood Dendritic": "cDC",
    "Mdc": "cDC",
    "Immature Dendritic Cells": "cDC",
    "Mature Dendritic": "cDC",

    "Nk": "NK CD56low",
    "Nk Cells-Peripheral Blood": "NK CD56low",
    "Purified Nk": "NK CD56low",
    "Natural Killer": "NK CD56low",
    "Expanded Nk": "NK CD56low",
    "Naive Nk": "NK CD56low",
    "Nk Cells.Il15": "NK CD56low",
    "Nk Cells.Il2": "NK CD56low",
    "Primary Nk": "NK CD56low",
}

cytokine_mapping = {
    "IFNA": "IFN-alpha1",
    "IFNB": "IFN-beta",
    "IFNG": "IFN-gamma",
    "GMCSF": "GM-CSF",
    "MCSF": "M-CSF",
    "IL1A": "IL-1-alpha",
    "IL1B": "IL-1-beta",
    "TNFA": "TNF-alpha",
    "IL10": "IL-10",
    "IL2": "IL-2",
    "IL3": "IL-3",
    "IL4": "IL-4",
    "IL6": "IL-6",
    "IL7": "IL-7",
    "IL13": "IL-13",
    "IL15": "IL-15",
    "IL21": "IL-21",
    "CD40L": "CD40L",
}

filtered_time_series['Mapped_Condition'] = filtered_time_series['Condition'].replace(cell_type_mapping)
filtered_time_series_in_hcd = filtered_time_series[filtered_time_series['Mapped_Condition'].isin(cell_type_mapping.values())].copy()
filtered_time_series_in_hcd['Mapped_Treatment'] = filtered_time_series_in_hcd['Treatment'].replace(cytokine_mapping)
filtered_time_series_in_hcd = filtered_time_series_in_hcd[filtered_time_series_in_hcd['Mapped_Treatment'].isin(cytokine_mapping.values())].copy()

In [ ]:
cell_type_mappping_series = pd.Series(cell_type_mapping).reset_index()
cell_type_mappping_series.columns = ['CytoSig', 'HCD']
cell_type_mappping_series = cell_type_mappping_series.groupby('HCD')['CytoSig'].agg(", ".join)
cell_type_mappping_series.to_excel(os.path.join(plot_folder, 'cytosig_to_hcd_cell_type_mapping.xlsx'))

In [ ]:
all_time_points = set()
for df in filtered_time_series_in_hcd['DataFrame']:
    all_time_points.update(df.columns)
all_time_points = sorted(all_time_points, key=lambda x: time_to_hours(x))

all_rs_by_time = []
for cytokine in filtered_time_series_in_hcd['Mapped_Treatment'].unique():
    curr_series = filtered_time_series_in_hcd[filtered_time_series_in_hcd['Mapped_Treatment'] == cytokine].copy()
    rs_by_time = []
    for _, row in curr_series.iterrows():
        cell_type_mapped = row['Mapped_Condition']
        df = row['DataFrame']

        hcd_subset = hcd[cell_type_mapped][cytokine].dropna()
        common_genes = df.index.intersection(hcd_subset.index)
        time_order = order_time_columns(df.columns)

        for time in time_order:
            x = hcd_subset.loc[common_genes]
            y = df[time].loc[common_genes]
            r = np.corrcoef(x, y)[0, 1]
            rs_by_time.append({
                "time": time,
                "r": r,
                "original_cell": row['Condition'],
                "cell_type": cell_type_mapped,
                "cytokine": cytokine,
                "study": row['origin_publication'],
            })
            
    rs_by_time = pd.DataFrame(rs_by_time)
    all_rs_by_time.append(rs_by_time)
    
all_rs_by_time_df = pd.concat(all_rs_by_time, ignore_index=True)
all_rs_by_time_df = all_rs_by_time_df.pivot_table(
    index=["cytokine", "cell_type", "original_cell", "study"],
    columns="time",
    values="r",
    fill_value=np.nan
)
all_rs_by_time_df = all_rs_by_time_df[all_time_points]

In [ ]:
from utils.unicode_utils import introduce_greek_letters

all_rs_by_time_df.reset_index(inplace=True)
all_rs_by_time_df['cytokine'] = all_rs_by_time_df['cytokine'].apply(introduce_greek_letters)
all_rs_by_time_df = all_rs_by_time_df.set_index(['cytokine', 'cell_type', 'original_cell', 'study'])

In [ ]:
# remove_cytokines = ['CD40L', 'IL-21']
# all_rs_by_time_df = all_rs_by_time_df[~all_rs_by_time_df.index.get_level_values('cytokine').isin(remove_cytokines)].copy()

In [ ]:
CYTOSIG_CELL_ABBREV = {
    "Monocyte": "Mono",
    "Macrophage": "Mac",
    "Mdm": "MDM",
    "Monocyte Derived Macrophage": "MDM",
    "Monocyte-Dervied Macrophage": "MDM",
    "Immature Dendritic": "imm. DC",
    "Pbmc": "PBMC",
    "Pbmc.Complete_Responder": "PBMC (MS)",
    "Whole Blood": "whole blood",
    "Purified Peripheral Blood": "PB",
    "T Cd4": "CD4 T",
    "T Cd4h": "CD4 Th",
}

fig, ax = plt.subplots(figsize=(6, 3))

sns.heatmap(
    all_rs_by_time_df,
    cmap="Reds",
    vmin=0,
    vmax=0.7,
    square=True,
    cbar_kws={'label': 'Pearson r with HCD', 'shrink': 0.7},
    ax=ax
)

# Hide default tuple-like y labels
ax.set_yticklabels([])
ax.set_ylabel("")
ax.set_xlabel("")
ax.tick_params(axis='both', length=0)

# Make room on the left
plt.subplots_adjust(left=0.55)

# label columns: x positions in heatmap-column units (negative = left of the heatmap)
label_cols = {"cytokine": -19.5, "HCD": -16, "CytoSig": -11, "study": -5.6}

for y, (cytokine, cell_type, original_cell, study) in enumerate(all_rs_by_time_df.index):
    values = [cytokine, cell_type, CYTOSIG_CELL_ABBREV.get(original_cell, original_cell), study]
    for x, value in zip(label_cols.values(), values):
        ax.text(x, y + 0.5, value, ha="left", va="center", transform=ax.transData)

for header, x in label_cols.items():
    ax.text(x, -0.4, header, ha="left", va="bottom", transform=ax.transData, fontweight="bold")

# plt.title("Comparison of HCD with CytoSig time series", y=1.1, fontsize=7)
#plt.tight_layout()
plt.savefig(os.path.join(plot_folder, f"hcd_comparison_heatmap.svg"), dpi=300)
plt.show()

# Hcd comparison (independent of time)

In [ ]:
cytosig_meta = cytosig_meta_orig.copy()

print(len(cytosig_meta))
cytosig_meta['Condition_Mapped'] = cytosig_meta['Condition'].replace(cell_type_mapping)
cytosig_meta = cytosig_meta[cytosig_meta['Condition_Mapped'].isin(cell_type_mapping.values())].copy()

print(len(cytosig_meta))
cytosig_meta['Treatment_Mapped'] = cytosig_meta['Treatment'].replace(cytokine_mapping)
cytosig_meta = cytosig_meta[cytosig_meta['Treatment_Mapped'].isin(cytokine_mapping.values())].copy()

In [ ]:
index = 'IFNB@Condition:monocyte-derived macrophages@E-MTAB-1437.MicroArray.Transformed_data_file'
row = cytosig_meta.loc[index]
cell_type_mapped = row['Condition_Mapped']
cytokine_mapped = row['Treatment_Mapped']

hcd_subset = hcd[cell_type_mapped][cytokine_mapped].dropna()
series = cytosig_orig[index].dropna()

common_genes = hcd_subset.index.intersection(series.index)
x = hcd_subset.loc[common_genes]
y = series.loc[common_genes]

plt.figure(figsize=(4,3))
r = np.corrcoef(x, y)[0, 1]
plt.scatter(x, y, s=3)
plt.xlabel("HCD log2FC")
plt.ylabel("CytoSig log2FC")
plt.title(f"Correlation: {r:.2f}")
plt.tight_layout()
plt.show()

In [ ]:
all_rs = []
for index, row in cytosig_meta.iterrows():
    cell_type_mapped = row['Condition_Mapped']
    cytokine_mapped = row['Treatment_Mapped']
    
    hcd_subset = hcd[cell_type_mapped][cytokine_mapped].dropna()
    series = cytosig_orig[index].dropna()
    
    common_genes = hcd_subset.index.intersection(series.index)
    if len(common_genes) < 1000:
            print(f"Not enough common genes ({len(common_genes)}), skipping.")
            continue
    x = hcd_subset.loc[common_genes]
    y = series.loc[common_genes]
    substantial_changes = (y.abs() > 1).sum()
    if substantial_changes < 100:
        continue
    
    r = np.corrcoef(x, y)[0, 1]
    all_rs.append({
        "r": r,
        "cell_type": cell_type_mapped,
        "cytokine": cytokine_mapped,
    })
    
all_rs_df_hcd = pd.DataFrame(all_rs)

In [ ]:
import itertools
same_source_groups = [
    {"GSE15390", "GSE17199", "GSE17946"}, # Schultze lab; GSE17199 samples are a subset of GSE15390
    {"GSE59184", "GSE43700"}, # Modlin lab, IL-10 on adherent PBMC
    {"GSE17974", "GSE32959"}, # Lahesmaa lab, IL-4 Th2 differentiation of cord-blood CD4 T cells
    {"GSE95588", "GSE129210"}, # HSS (Donlin / Ivashkiv), TNF-stimulated macrophages
    {"GSE120084", "GSE120086"}, # same submitter, Ad5f35-transduced macrophages 
]
def same_source(s1, s2):
    return s1 == s2 or any(s1 in g and s2 in g for g in same_source_groups)

all_rs = []
grouped_indices = cytosig_meta.groupby(['Condition_Mapped', 'Treatment_Mapped']).groups
for (cell_type_mapped, cytokine_mapped), indices in grouped_indices.items():
    non_redunant_combs = itertools.combinations(indices, 2)
    for idx1, idx2 in non_redunant_combs:
        study1 = cytosig_meta.loc[idx1, 'origin_publication']
        study2 = cytosig_meta.loc[idx2, 'origin_publication']
        if same_source(study1, study2):
            continue
        
        series1 = cytosig_orig[idx1].dropna()
        series2 = cytosig_orig[idx2].dropna()

        common_genes = series1.index.intersection(series2.index)
        if len(common_genes) < 1000:
            print(f"Not enough common genes ({len(common_genes)}) between {idx1} and {idx2}, skipping.")
            continue
        x = series1.loc[common_genes]
        y = series2.loc[common_genes]

        substantial_changes_1 = (x.abs() > 1).sum()
        substantial_changes_2 = (y.abs() > 1).sum()
        if substantial_changes_1 < 100 or substantial_changes_2 < 100:
            # print(f"Not enough substantial changes between {idx1} and {idx2} ({substantial_changes_1} and {substantial_changes_2}), skipping.")
            continue

        r = np.corrcoef(x, y)[0, 1]
        all_rs.append({
            "r": r,
            "cell_type": cell_type_mapped,
            "cytokine": cytokine_mapped,
            "idx1": idx1,
            "idx2": idx2,
        })
all_rs_df_cytosig = pd.DataFrame(all_rs)

In [ ]:
def order_df_by_col(df, col, order):
    df = df.copy()
    df['_order'] = pd.Categorical(df[col], categories=order, ordered=True)
    df = df.sort_values('_order').drop(columns='_order').reset_index(drop=True)
    return df

grouped_rs_hcd = all_rs_df_hcd.groupby(['cell_type', 'cytokine'])['r'].agg(['mean', 'std', 'count']).reset_index()
grouped_rs_cytosig = all_rs_df_cytosig.groupby(['cell_type', 'cytokine'])['r'].agg(['mean', 'std', 'count']).reset_index()

present_cell_types = set(grouped_rs_hcd['cell_type']).intersection(set(grouped_rs_cytosig['cell_type']))
present_cytokines = set(grouped_rs_hcd['cytokine']).intersection(set(grouped_rs_cytosig['cytokine']))
present_combinations = set(zip(grouped_rs_hcd['cytokine'], grouped_rs_hcd['cell_type'])).intersection(
    set(zip(grouped_rs_cytosig['cytokine'], grouped_rs_cytosig['cell_type']))
)

cell_type_order = list(CELL_TYPE_COLORS.keys()) + ["PBMC"]
ordered_cell_types = [cell for cell in cell_type_order if cell in present_cell_types]
ordered_cytokines = [cyto for cyto in CYTOKINE_ORDER if cyto in present_cytokines]

ordered_combinations = []
for cell in ordered_cell_types:
    for cyto in ordered_cytokines:
        if (cyto, cell) in present_combinations:
            ordered_combinations.append((cyto, cell))
            
grouped_rs_hcd['combination'] = list(zip(grouped_rs_hcd['cytokine'], grouped_rs_hcd['cell_type']))
grouped_rs_cytosig['combination'] = list(zip(grouped_rs_cytosig['cytokine'], grouped_rs_cytosig['cell_type']))

grouped_rs_hcd = order_df_by_col(grouped_rs_hcd, 'combination', ordered_combinations)
grouped_rs_cytosig = order_df_by_col(grouped_rs_cytosig, 'combination', ordered_combinations)

In [ ]:
plot_df = pd.concat([
    all_rs_df_hcd.assign(source="HCD"),
    all_rs_df_cytosig.assign(source="CytoSig"),
], ignore_index=True)

replacement_dict = {
    "HCD": "CytoSig vs HCD",
    "CytoSig": "CytoSig vs CytoSig",
}

plot_df["source"] = plot_df["source"].replace(replacement_dict)
plot_df["cytokine_raw"] = plot_df["cytokine"]
plot_df["cytokine_label"] = plot_df["cytokine_raw"].apply(introduce_greek_letters)
plot_df["combination"] = list(zip(plot_df["cytokine_raw"], plot_df["cell_type"]))

observed_combinations = set(plot_df["combination"])
observed_ordered_combinations = [
    combo for combo in ordered_combinations
    if combo in observed_combinations
]
plot_df = plot_df[plot_df["combination"].isin(observed_ordered_combinations)].copy()
combo_to_label = {
    combo: f"{combo[1]}__{introduce_greek_letters(combo[0])}"
    for combo in observed_ordered_combinations
}
label_order = [
    combo_to_label[combo]
    for combo in observed_ordered_combinations
]
plot_df["combo_label"] = plot_df["combination"].map(combo_to_label)

In [ ]:
fig, ax = plt.subplots(figsize=(7, 2))

sns.stripplot(
    data=plot_df,
    x="combo_label",
    y="r",
    hue="source",
    order=label_order,
    dodge=True,
    alpha=1,
    size=2,
    # color="black",
    legend=True,
    ax=ax,
)

cytokine_labels = [introduce_greek_letters(cyto) for cyto, cell in observed_ordered_combinations]
ax.set_xticklabels(cytokine_labels, rotation=90)

# add grouped cell-type labels below
for cell_type in ordered_cell_types:
    positions = [i for i, (cyto, cell) in enumerate(observed_ordered_combinations) if cell == cell_type]
    if not positions:
        continue

    start, end = min(positions), max(positions)
    mid = (start + end) / 2

    ax.text(
        mid,
        -0.45,
        cell_type,
        ha="center",
        va="top",
        transform=ax.get_xaxis_transform(),
        fontsize=7,
    )

    ax.plot(
        [start - 0.4, end + 0.4],
        [-0.375, -0.375],
        transform=ax.get_xaxis_transform(),
        color="black",
        linewidth=0.75,
        clip_on=False,
    )

    ax.axvline(end + 0.5, color="black", linewidth=0.5, alpha=0.2)

ax.set_xlabel("")
ax.set_ylabel("Pearson r")
ax.tick_params(axis="x", length=0)

# remove duplicate legend entries
handles, labels = ax.get_legend_handles_labels()
ax.legend(handles[:2], labels[:2], title="", loc="upper left")
plt.xlim(-0.5, len(label_order) - 0.5)
plt.ylim(-0.25, 1)
plt.tight_layout()
plt.subplots_adjust(bottom=0.45)
plt.savefig(os.path.join(plot_folder, f"hcd_cytosig_comparison_boxplot.svg"), dpi=300)
plt.show()